\REQUETE SQL ROMAN - VENTE + LOGISTIQUE

In [ ]:
-- PARTIE VENTE






-- 1) Chiffre d’affaires par mois et par région + taux d’évolution mensuel : Suivre les revenus générés par région et par mois pour identifier les tendances géographiques.



-- CA PAR MOIS ET PAR REGIONS

SELECT
    DATE_FORMAT(o.orderDate, '%Y-%m') AS 'year_month',
    COALESCE(off.territory, 'Unknown') AS region,
    SUM(od.quantityOrdered * od.priceEach) AS total_revenue
FROM orders o
JOIN orderdetails od ON o.orderNumber = od.orderNumber
JOIN customers cu ON o.customerNumber = cu.customerNumber
LEFT JOIN employees emp ON cu.salesRepEmployeeNumber = emp.employeeNumber
LEFT JOIN offices off ON off.officeCode = emp.officeCode
GROUP BY DATE_FORMAT(o.orderDate, '%Y-%m'), COALESCE(off.territory, 'Unknown')
ORDER BY 'year_month', region;

In [ ]:
-- 2) Produits les plus/moins vendus par catégorie : Identifier les produits les plus performants dans chaque catégorie.



-- CA PAR PRODUIT ET QUANTITE DE PRODUIT
SELECT
  ps.productLine,
  SUM(ps.total_qty)     AS qty_by_line,
  SUM(ps.total_revenue) AS revenue_by_line
FROM (
  SELECT
    p.productLine,
    p.productCode,
    p.productName,
    SUM(od.quantityOrdered)                AS total_qty,
    SUM(od.quantityOrdered * od.priceEach) AS total_revenue
  FROM orderdetails od
  JOIN orders   o ON o.orderNumber   = od.orderNumber
  JOIN products p ON p.productCode   = od.productCode
  GROUP BY p.productLine, p.productCode, p.productName
) ps
GROUP BY ps.productLine
ORDER BY revenue_by_line DESC;

In [ ]:
-- PRODUITS LES PLUS / MOINS VENDUS PAR CATÉGORIE

-- LES PLUS VENDUS PAR CAT :

SELECT *
FROM (
  SELECT
    p.productLine,
    p.productCode,
    p.productName,
    SUM(od.quantityOrdered * od.priceEach) AS CA,
    ROW_NUMBER() OVER (PARTITION BY p.productLine ORDER BY SUM(od.quantityOrdered * od.priceEach) DESC) AS rang
  FROM orderdetails od
  JOIN orders o ON o.orderNumber = od.orderNumber
  JOIN products p ON p.productCode = od.productCode
  GROUP BY p.productLine, p.productCode, p.productName
) t
WHERE rang <= 5
ORDER BY productLine, rang;

In [ ]:
--LES MOINS VENDU PAR CAT :

SELECT *
FROM (
  SELECT
    p.productLine,
    p.productCode,
    p.productName,
    SUM(od.quantityOrdered * od.priceEach) AS CA,
    ROW_NUMBER() OVER (
      PARTITION BY p.productLine
      ORDER BY SUM(od.quantityOrdered * od.priceEach) ASC   --  ici on passe en ASC
    ) AS rang
  FROM orderdetails od
  JOIN orders o ON o.orderNumber = od.orderNumber
  JOIN products p ON p.productCode = od.productCode
  GROUP BY p.productLine, p.productCode, p.productName
) t
WHERE rang <= 5
ORDER BY productLine, rang;

In [ ]:
-- 3) La marge brute par produit et par catégorie : Mesurer la marge brute et en déduire les produits/catégories les plus/moins rentables.


-- CA Total par catégorie pour recouper visuellement

WITH product_sales AS (
  SELECT
    p.productLine,
    p.productCode,
    p.productName,
    SUM(od.quantityOrdered * od.priceEach) AS total_sales
  FROM orderdetails od
  JOIN orders   o ON o.orderNumber  = od.orderNumber
  JOIN products p ON p.productCode  = od.productCode
  GROUP BY p.productLine, p.productCode, p.productName
)
SELECT productLine, SUM(total_sales) AS ca_par_categorie
FROM product_sales
GROUP BY productLine
ORDER BY ca_par_categorie DESC;

In [ ]:
-- Ajouter la quantité totale et la marge de produit vendu

SELECT
  p.productLine,
  p.productCode,
  p.productName,
  SUM(od.quantityOrdered) AS total_quantity,
  SUM(od.quantityOrdered * od.priceEach) AS total_sales,
  SUM((od.priceEach - p.buyPrice) * od.quantityOrdered) AS total_margin
FROM orderdetails od
JOIN orders o   ON o.orderNumber = od.orderNumber
JOIN products p ON p.productCode = od.productCode
GROUP BY p.productLine, p.productCode, p.productName;

In [ ]:
-- VENTE MENSUELLES PAR CAT + LAG PAR MOIS
-- A) Brique "ventes mensuelles par catégorie"
SELECT
  DATE_FORMAT(o.orderDate, '%Y-%m-01') AS month_start,   -- clé mois (1er du mois)
  p.productLine,
  SUM(od.quantityOrdered * od.priceEach) AS sales
FROM orderdetails od
JOIN orders   o ON o.orderNumber  = od.orderNumber
JOIN products p ON p.productCode  = od.productCode
GROUP BY DATE_FORMAT(o.orderDate, '%Y-%m-01'), p.productLine;

In [ ]:
-- Étape avec LAG et Month over Month (evolution mensuel)

SELECT
  productLine,
  month_start,
  sales AS sales_cur,
  LAG(sales) OVER (
      PARTITION BY productLine
      ORDER BY month_start
  ) AS sales_prev,
  CASE
    WHEN LAG(sales) OVER (PARTITION BY productLine ORDER BY month_start) IS NULL
         OR LAG(sales) OVER (PARTITION BY productLine ORDER BY month_start) = 0
      THEN NULL
    ELSE (sales - LAG(sales) OVER (PARTITION BY productLine ORDER BY month_start))
         / LAG(sales) OVER (PARTITION BY productLine ORDER BY month_start)
  END AS mom_pct
FROM (
  SELECT
    DATE_FORMAT(o.orderDate, '%Y-%m-01') AS month_start,   -- clé mois (1er du mois)
    p.productLine,
    SUM(od.quantityOrdered * od.priceEach) AS sales
  FROM orderdetails od
  JOIN orders   o ON o.orderNumber = od.orderNumber
  JOIN products p ON p.productCode = od.productCode
  GROUP BY DATE_FORMAT(o.orderDate, '%Y-%m-01'), p.productLine
) AS monthly
ORDER BY productLine, month_start;

In [ ]:
-- PARTIE LOGISTIQUE

-- 1/ -- Calcul stock faible --
SELECT productName, productCode, productLine, ProductScale, productVendor, quantityInStock
FROM products
WHERE quantityInStock <10;

-- 2) Durée moyenne de traitement des commandes + commandes au-dessus de la moyenne de livraison :
--    Mesurer l’efficacité opérationnelle en analysant le temps entre la date de commande et la date d’expédition.


WITH processing AS (
  SELECT
    o.orderNumber,
    DATEDIFF(o.shippedDate, o.orderDate) AS processing_time, -- DATEDIFF = la différence en jours entre la commande et l’expédition (temps de traitement)
    cu.salesRepEmployeeNumber AS sales_rep                   -- identifiant du commercial responsable
  FROM orders o
  JOIN customers cu ON o.customerNumber = cu.customerNumber
  WHERE o.shippedDate IS NOT NULL   -- on garde seulement les commandes expédiées
)
SELECT
  e.employeeNumber,
  CONCAT(e.firstName, ' ', e.lastName) AS sales_rep_name,  -- crée le nom complet du commercial avec CONCAT
  ROUND(AVG(p.processing_time), 2) AS avg_processing_time,  -- durée moyenne en jours et arrondit à 2 décimales (ROUND)
  SUM(CASE                                                  -- compte combien de commandes de ce commercial ont mis plus de temps que la moyenne globale
        WHEN p.processing_time > (SELECT AVG(processing_time) FROM processing) THEN 1  -- calcule la moyenne globale pour comparaison et attribut +1 ou 0 au 'compteur' above_avg
        ELSE 0
      END) AS above_avg_orders                               -- nb de commandes plus lentes que la moyenne (+1 ou +0 en fonction de la moyenne au dessus)
FROM processing p
JOIN employees e ON p.sales_rep = e.employeeNumber
GROUP BY e.employeeNumber, e.firstName, e.lastName -- Regroupe par commercial
ORDER BY avg_processing_time ASC; -- trie du plus rapide au plus lent


In [ ]:
REQUETE SQL ROMAIN FINANCE

In [ ]:
🔴 Clients générant le plus/moins de revenus : Identifier les clients générant le plus de revenus pour mieux les fidéliser.

SELECT
c.customerName AS nom_client,
c.customerNumber AS numéro_client,
SUM(quantityOrdered * priceEach) AS total_revenue

FROM customers c
JOIN orders o
	ON c.customerNumber = o.customerNumber
JOIN orderdetails od
	ON od.orderNumber = o.orderNumber
GROUP BY
c.customerName,
c.customerNumber
ORDER BY total_revenue DESC;

In [ ]:
    🟢 Taux de recouvrement des créances par client : Identifier les clients ayant un montant élevé de commandes non payées.

SELECT
    c.customerNumber AS numéro_client,
    c.customerName AS nom_client,
SUM(od.quantityOrdered * od.priceEach) AS total_commandes,
COALESCE(SUM(DISTINCT p.amount), 0) AS total_paiements,
(SUM(od.quantityOrdered * od.priceEach) - COALESCE(SUM(DISTINCT p.amount), 0)) AS montant_impayé

FROM customers c
JOIN orders o
    ON c.customerNumber = o.customerNumber
JOIN orderdetails od
    ON o.orderNumber = od.orderNumber
LEFT JOIN payments p
    ON c.customerNumber = p.customerNumber

GROUP BY c.customerNumber, c.customerName
ORDER BY montant_impayé DESC;


-- Total des commandes par client

WITH commandes AS (
    SELECT
        c.customerNumber,
        c.customerName,
        ROUND(SUM(od.quantityOrdered * od.priceEach), 2) AS total_commandes
    FROM customers c
    JOIN orders o ON c.customerNumber = o.customerNumber
    JOIN orderdetails od ON o.orderNumber = od.orderNumber
    GROUP BY c.customerNumber, c.customerName
),

-- Total des paiements par client

paiements AS (
    SELECT
        p.customerNumber,
        ROUND(SUM(p.amount), 2) AS total_paiements
    FROM payments p
    GROUP BY p.customerNumber
)

-- Regroupement final avec taux de recouvrement

SELECT
    c.customerNumber AS numero_client,
    c.customerName AS nom_client,
    commandes.total_commandes,
    IFNULL(paiements.total_paiements, 0) AS total_paiements,
    ROUND(commandes.total_commandes - IFNULL(paiements.total_paiements, 0), 2) AS montant_impaye,
    ROUND((IFNULL(paiements.total_paiements, 0) / commandes.total_commandes) * 100, 2) AS taux_recouvrement
FROM commandes
LEFT JOIN paiements ON commandes.customerNumber = paiements.customerNumber
JOIN customers c ON c.customerNumber = commandes.customerNumber
ORDER BY montant_impaye DESC;


In [ ]:
REQUETES CELINE RESSOURCES HUMAINES + LOGISTIQUE

In [ ]:
-- Chiffre d'Affaires: prix de vente * quantités vendues

SELECT SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires_total
FROM orderdetails ;

-- Chiffres d'Affaires par représentant commercial--

SELECT
    emp.firstName,
    emp.lastName,
    emp.employeeNumber AS salesRepEmployeeNumber,
    emp.jobTitle,
    SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires
FROM
    employees emp
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
    emp.firstName,
    emp.lastName,
    emp.employeeNumber,
    emp.jobTitle
ORDER BY
    Chiffre_d_Affaires DESC;

    -- Chiffres d'Affaires par bureau --

SELECT offices.officeCode AS Numero_de_bureau, offices.country, territory, SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires
FROM offices
JOIN
	employees emp ON offices.officeCode = emp.officeCode
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
   Numero_de_bureau
ORDER BY
    Chiffre_d_Affaires DESC;

-- Chiffre d'Affaires par territoire --

SELECT territory , SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires
FROM offices
JOIN
	employees emp ON offices.officeCode = emp.officeCode
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
   territory
ORDER BY
    Chiffre_d_Affaires DESC;


    -- Requête ratio commandes/paiements --

WITH ca_par_commercial AS (
SELECT
    emp.employeeNumber,
    SUM(quantityOrdered*priceEach) AS chiffre_d_affaires
FROM
    employees emp
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
    emp.employeeNumber),
paiements_par_commercial AS (
SELECT
	emp.employeeNumber,
    SUM(pay.amount) AS total_payments
FROM
    employees emp
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
    payments pay ON cust.customerNumber = pay.customerNumber
GROUP BY
	emp.employeeNumber)
SELECT
lastName,
firstName,
emp.employeeNumber,
ca.chiffre_d_affaires / NULLIF(p.total_payments,0) AS ratio
FROM
employees emp
LEFT JOIN ca_par_commercial ca ON emp.employeeNumber = ca.employeeNumber
LEFT JOIN paiements_par_commercial p ON emp.employeeNumber = p.employeeNumber
WHERE ca.chiffre_d_affaires IS NOT NULL
ORDER BY ratio DESC ;



-- Pourcentage du CA du Commercial sur le CA du Bureau --

 WITH CA_par_commercial AS
(
   SELECT
    emp.firstName,
    emp.lastName,
    emp.employeeNumber AS salesRepEmployeeNumber,
    emp.jobTitle,
    emp.officeCode,
    SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires_commercial
FROM
    employees emp
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
    emp.firstName,
    emp.lastName,
    emp.employeeNumber,
    emp.jobTitle, emp.officeCode
    ),
    CA_par_bureau AS
    (
    SELECT
    offices.officeCode AS Numero_de_bureau,
    offices.country,
    territory,
    SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires_bureau
FROM offices
JOIN
	employees emp ON offices.officeCode = emp.officeCode
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
   Numero_de_bureau
   )
SELECT
    CA_par_commercial .firstName,
    CA_par_commercial .lastName,
    CA_par_commercial .salesRepEmployeeNumber,
    CA_par_commercial .jobTitle,
    CA_par_commercial .officeCode,
    CA_par_bureau.country,
    CA_par_bureau.territory,
    CA_par_commercial.Chiffre_d_Affaires_commercial,
    CA_par_bureau.Chiffre_d_Affaires_bureau,
    ROUND((CA_par_commercial.Chiffre_d_Affaires_commercial/ CA_par_bureau.Chiffre_d_Affaires_bureau) * 100, 2) AS Pourcentage_de_CA
FROM CA_par_commercial
JOIN
	CA_par_bureau ON CA_par_commercial.officeCode= CA_par_bureau.Numero_de_bureau
ORDER BY
   Pourcentage_de_CA DESC;

   -- Pourcentage CA Bureau sur CA total--

WITH CA_par_bureau AS
    (
    SELECT
    offices.officeCode AS Numero_de_bureau,
    offices.country,
    territory,
    SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires_bureau
FROM offices
JOIN
	employees emp ON offices.officeCode = emp.officeCode
JOIN
    customers cust ON emp.employeeNumber = cust.salesRepEmployeeNumber
JOIN
	orders o ON cust.customerNumber= o.customerNumber
JOIN
    orderdetails ord ON o.orderNumber = ord.orderNumber
GROUP BY
   Numero_de_bureau
   ),
CA_total AS(
SELECT SUM(quantityOrdered*priceEach) AS Chiffre_d_Affaires_total
FROM orderdetails
)
SELECT
Numero_de_bureau,
CA_par_bureau.country,
CA_par_bureau.territory,
Chiffre_d_Affaires_bureau,
ROUND((CA_par_bureau.Chiffre_d_Affaires_bureau/ CA_total.Chiffre_d_Affaires_total)*100,2) AS pourcentage_CA_Bureau
FROM CA_par_bureau
CROSS JOIN CA_total
ORDER BY pourcentage_CA_Bureau DESC;


 -- Nouvelle requête Etat du stock--

SELECT productName, productCode, productLine, ProductScale, productVendor, quantityInStock,
CASE
    WHEN quantityInStock < 10 THEN 'Stock Faible'
    WHEN quantityInStock > 10 AND quantityInStock < 30 THEN 'Stock Moyen'
    ELSE 'Stock OK'
END AS etat_du_stock
FROM products;

--  Temps de traitement des commandes de la date de commande à la date d'expédition --

SELECT
orderNumber, orderDate, shippedDate, status,
datediff(shippedDate, orderDate) AS temps_de_traitement
FROM orders
ORDER BY orderNumber;

-- Durée moyenne de traitement des commandes --

SELECT AVG(datediff(shippedDate, orderDate)) AS duree_moyenne_de_traitement
FROM orders;

-- Durée moyenne de traitement des commandes par statut des commandes--

SELECT status, AVG(datediff(shippedDate, orderDate)) AS duree_moyenne_de_traitement
FROM orders
GROUP BY status;

-- commandes au-dessus de la moyenne de livraison --

WITH calcul_du_temps_de_traitement AS (
SELECT
orderNumber, orderDate, shippedDate, status,
datediff(shippedDate, orderDate) AS temps_de_traitement
FROM orders
)
   SELECT orderNumber, orderDate, shippedDate, status, temps_de_traitement
   FROM calcul_du_temps_de_traitement
   WHERE temps_de_traitement >
         (
          SELECT AVG(temps_de_traitement)
		  FROM calcul_du_temps_de_traitement
          )
ORDER BY temps_de_traitement DESC;

